In [2]:
import pathlib
import re
import pandas as pd

CODEML_RUNS_DIR = pathlib.Path("codeml_runs")
GROUND_TRUTH_CSV = pathlib.Path("simulated_data/ground_truth.csv")

def parse_rho(mlb_path):
    """Extract inferred rho from a codeml mlb output file."""
    text = mlb_path.read_text()
    match = re.search(r'rho \(correlation\)\s*=\s*([0-9.]+)', text)
    return float(match.group(1)) if match else None

# Collect results from all completed runs
rows = []
for mlb in sorted(CODEML_RUNS_DIR.glob("*/mlb")):
    if mlb.stat().st_size == 0:
        continue
    rho = parse_rho(mlb)
    if rho is not None:
        rows.append({"sim_name": mlb.parent.name, "codeml_rho": rho})

codeml_df = pd.DataFrame(rows)

# Join with ground truth
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
results = codeml_df.merge(gt_df, on="sim_name")

print(f"Completed runs: {len(results)}")
results.head()

Completed runs: 333


,sim_name,codeml_rho,true_alpha,true_rho,true_tree_scale,n_taxa
0,sim_0001_a1.688_r0.789,0.81570,1.688,0.789,0.005116,20
1,sim_0002_a1.601_r0.685,0.20828,1.601,0.685,0.020834,20
2,sim_0003_a1.224_r0.32,0.37040,1.224,0.320,0.041679,20
3,sim_0004_a0.429_r0.718,0.70245,0.429,0.718,0.082831,20
4,sim_0005_a1.695_r0.505,0.17051,1.695,0.505,0.014964,20


In [ ]:
import pathlib
import time
import re
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../sbi_pipeline")  # adjust to where features_calculator.py lives
from features_calculator import calculate_cnn_input

CODEML_RUNS_DIR = pathlib.Path("codeml_runs")
GROUND_TRUTH_CSV = pathlib.Path("simulated_data/ground_truth.csv")
SIMULATED_DATA_DIR = pathlib.Path("simulated_data")
POST_PATH = "../sbi_pipeline/sbi_models/posterior.pt"

# ── model def (must match training) ─────────────────────────────────────────
class CNN1dEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.ReLU(),
        )
    def forward(self, x):
        return self.net(x).mean(dim=-1)

# ── helpers ───────────────────────────────────────────────────────────────────
def parse_rho(mlb_path):
    text = mlb_path.read_text()
    match = re.search(r'rho \(correlation\)\s*=\s*([0-9.]+)', text)
    return float(match.group(1)) if match else None

def parse_codeml_time(mlb_path):
    """Parse runtime from mlb file e.g. 'Time used:  1:25' -> seconds."""
    text = mlb_path.read_text()
    match = re.search(r'Time used:\s+(\d+):(\d+)', text)
    if match:
        return int(match.group(1)) * 60 + int(match.group(2))
    return None

def read_phy(phy_path):
    lines = phy_path.read_text().strip().splitlines()
    seqs = []
    for line in lines[1:]:
        parts = line.split()
        if len(parts) >= 2:
            seqs.append(parts[1])
    return seqs

# ── find successful codeml runs ───────────────────────────────────────────────
codeml_rows = []
for mlb in sorted(CODEML_RUNS_DIR.glob("*/mlb")):
    if mlb.stat().st_size == 0:
        continue
    rho = parse_rho(mlb)
    codeml_time = parse_codeml_time(mlb)
    if rho is not None:
        codeml_rows.append({"sim_name": mlb.parent.name, "codeml_rho": rho, "codeml_time": codeml_time})

codeml_df = pd.DataFrame(codeml_rows)
print(f"Found {len(codeml_df)} successful codeml runs")

# ── load posterior ────────────────────────────────────────────────────────────
posterior = torch.load(POST_PATH, map_location="cpu", weights_only=False)

# ── run SBI on each sim ───────────────────────────────────────────────────────
sbi_rows = []
for sim_name in codeml_df["sim_name"]:
    phy_path = SIMULATED_DATA_DIR / sim_name / "alignment.phy"
    if not phy_path.exists():
        print(f"  WARNING: alignment not found for {sim_name}, skipping")
        continue

    seqs = read_phy(phy_path)
    seqs = [s[:500] for s in seqs]

    x_obs = calculate_cnn_input(seqs, alphabet_size=20).unsqueeze(0)
    t0 = time.time()
    with torch.no_grad():
        samples = posterior.sample((1000,), x=x_obs)
    sbi_elapsed = time.time() - t0

    rho_median = samples[:, 1].median().item()
    rho_std    = samples[:, 1].std().item()

    sbi_rows.append({"sim_name": sim_name, "sbi_rho": rho_median, "sbi_rho_std": rho_std, "sbi_time": sbi_elapsed})

sbi_df = pd.DataFrame(sbi_rows)

# ── join everything ───────────────────────────────────────────────────────────
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
results = codeml_df.merge(sbi_df, on="sim_name").merge(gt_df, on="sim_name")
print(f"\nComparison ready: {len(results)} runs")

# ── scatter plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in zip(axes, ["codeml_rho", "sbi_rho"], ["codeml", "SBI"]):
    ax.scatter(results["true_rho"], results[col], alpha=0.6, s=20)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("true rho")
    ax.set_ylabel(f"{label} rho")
    ax.set_title(label)
plt.tight_layout()
plt.show()

# ── runtime boxplot ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot(
    [results["codeml_time"].dropna(), results["sbi_time"]],
    labels=["codeml", "SBI"],
    showfliers=True,
)
ax.set_ylabel("time (seconds)")
ax.set_title("Runtime comparison")
plt.tight_layout()
plt.show()

KeyError: 'sim_name'